In [1]:
import cv2
import json
import numpy as np
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp
from tqdm import tqdm

In [2]:
import json
json_path = '../dataset/WLASL_v0.3.json' 
try:
    with open(json_path, 'r') as file:
        wlasl_data = json.load(file)
    print(f"Total words in dataset: {len(wlasl_data)}")
    print("First 10 words found:")
    for i in range(10):
        print(f"- {wlasl_data[i]['gloss']}")
except FileNotFoundError:
    print("Could not find the JSON file. Check the folder path and name.")

Total words in dataset: 2000
First 10 words found:
- book
- drink
- computer
- before
- chair
- go
- clothes
- who
- candy
- cousin


### MediaPipe Holistic Landmarks (Face, Pose, Hands)
* FACE_LANDMARKS = 468 * (x, y, z) = 1404 values
* POSE_LANDMARKS = 33 * (x, y, z, vis) = 132  values
* LEFT_HAND = 21 * (x, y, z) = 63   values
* RIGHT_HAND = 21 * (x, y, z) = 63   values

Total landmarks = 543\
Total values = 1404 + 132 + 63 + 63 = 1662

In [3]:
with open('../dataset/nslt_100.json') as f:
    NSLT = json.load(f)

with open('../dataset/WLASL_v0.3.json') as f:
    WLASL = json.load(f)

In [4]:
# video_id → gloss metadata (fps, trim points, bbox)
WLASL_META = {}
for entry in WLASL:
    for inst in entry['instances']:
        WLASL_META[inst['video_id']] = {
            'gloss'      : entry['gloss'],        # (book, drink, computer, ...)
            'fps'        : inst['fps'],           # always 25 in this dataset
            'frame_start': inst['frame_start'],   # 1-indexed, some videos trimmed
            'frame_end'  : inst['frame_end'],     # -1 means use full video
            'bbox'       : inst['bbox'],          # [x1, y1, x2, y2]
        }

# class_idx → gloss name  (derived from nslt + wlasl together)
CLASS_TO_GLOSS = {}
for vid_id, val in NSLT.items():
    cls = val['action'][0]
    if vid_id in WLASL_META and cls not in CLASS_TO_GLOSS:
        CLASS_TO_GLOSS[cls] = WLASL_META[vid_id]['gloss']

GLOSS_TO_CLASS = {v: k for k, v in CLASS_TO_GLOSS.items()}

In [5]:
VIDEO_DIR    = '../dataset/videos/'      # folder with 00000.mp4 etc.
OUTPUT_DIR   = '../dataset/processed/'   # where .npy files will be saved
NUM_FRAMES   = 30                        # frames per sequence
NUM_CLASSES  = 100                       # from nslt_100 — exactly 100 classes, indices 0–99
FPS          = 25                        # confirmed from WLASL_v0_3.json — all videos are 25fps
USE_BBOX     = True                      # crop to signer bbox before MediaPipe (improves detection)

In [6]:
TRAIN_IDS = [vid for vid, val in NSLT.items() if val['subset'] == 'train']  # 1442 videos
VAL_IDS   = [vid for vid, val in NSLT.items() if val['subset'] == 'val']    # 338  videos
TEST_IDS  = [vid for vid, val in NSLT.items() if val['subset'] == 'test']   # 258  videos

print(f"Classes : {NUM_CLASSES}")
print(f"Train   : {len(TRAIN_IDS)} | Val: {len(VAL_IDS)} | Test: {len(TEST_IDS)}")
print(f"Total   : {len(NSLT)} videos")
print(f"Sample class map: { {k: CLASS_TO_GLOSS[k] for k in range(10)} }")

Classes : 100
Train   : 1442 | Val: 338 | Test: 258
Total   : 2038 videos
Sample class map: {0: 'book', 1: 'drink', 2: 'computer', 3: 'before', 4: 'chair', 5: 'go', 6: 'clothes', 7: 'who', 8: 'candy', 9: 'cousin'}


In [7]:
def extract_keypoints(result):
    # pose_landmarks is a flat List[NormalizedLandmark] — no [0] needed
    pose = np.array([
        [lm.x, lm.y, lm.z, lm.visibility]
        for lm in result.pose_landmarks
    ]).flatten() if result.pose_landmarks else np.zeros(132)

    lh = np.array([
        [lm.x, lm.y, lm.z]
        for lm in result.left_hand_landmarks
    ]).flatten() if result.left_hand_landmarks else np.zeros(63)

    rh = np.array([
        [lm.x, lm.y, lm.z]
        for lm in result.right_hand_landmarks
    ]).flatten() if result.right_hand_landmarks else np.zeros(63)

    return np.concatenate([pose, lh, rh])  # 258-dim

In [8]:
def normalize_keypoints(keypoints):
    lh_start = 132
    rh_start = 132 + 63

    # Left hand
    if not np.allclose(keypoints[lh_start:lh_start + 63], 0):
        lh   = keypoints[lh_start:lh_start + 63].reshape(21, 3)
        lh   = lh - lh[0]  # subtract wrist
        span = np.linalg.norm(lh.max(axis=0) - lh.min(axis=0)) + 1e-6
        keypoints[lh_start:lh_start + 63] = (lh / span).flatten()

    # Right hand
    if not np.allclose(keypoints[rh_start:rh_start + 63], 0):
        rh   = keypoints[rh_start:rh_start + 63].reshape(21, 3)
        rh   = rh - rh[0]
        span = np.linalg.norm(rh.max(axis=0) - rh.min(axis=0)) + 1e-6
        keypoints[rh_start:rh_start + 63] = (rh / span).flatten()

    return keypoints

In [9]:
_TIMESTAMP_MS = 0

def process_video(video_id):
    global _TIMESTAMP_MS

    meta       = WLASL_META[video_id]
    video_path = os.path.join(VIDEO_DIR, f'{video_id}.mp4')

    if not os.path.exists(video_path):
        return None

    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0:
        cap.release()
        return None

    f_start         = meta['frame_start'] - 1
    f_end           = total if meta['frame_end'] == -1 else meta['frame_end']
    f_end           = min(f_end, total)
    indices         = np.linspace(f_start, f_end - 1, NUM_FRAMES, dtype=int)
    x1, y1, x2, y2 = meta['bbox']

    # Fresh landmarker per video — avoids segmentation smoother
    # dimension mismatch between videos with different bbox sizes
    options = vision.HolisticLandmarkerOptions(
        base_options=python.BaseOptions(
            model_asset_path='../models/holistic_landmarker.task'
        ),
        running_mode=vision.RunningMode.VIDEO,
        min_face_detection_confidence=0.5,
        min_pose_detection_confidence=0.5,
        min_hand_landmarks_confidence=0.5,
        output_segmentation_mask=False,   # disable segmentation entirely
    )

    sequence = []

    with vision.HolisticLandmarker.create_from_options(options) as landmarker:
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()

            if not ret or frame is None:
                sequence.append(sequence[-1] if sequence else np.zeros(258))
                _TIMESTAMP_MS += 1
                continue

            if USE_BBOX:
                h, w    = frame.shape[:2]
                cx1     = max(0, min(x1, w - 1))
                cy1     = max(0, min(y1, h - 1))
                cx2     = max(cx1 + 1, min(x2, w))
                cy2     = max(cy1 + 1, min(y2, h))
                cropped = frame[cy1:cy2, cx1:cx2]
                frame   = cropped if cropped.size > 0 else frame

            rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            _TIMESTAMP_MS += 1
            result = landmarker.detect_for_video(mp_image, _TIMESTAMP_MS)

            kp = extract_keypoints(result)
            kp = normalize_keypoints(kp)
            sequence.append(kp)

    cap.release()
    return np.array(sequence, dtype=np.float32)  # (30, 258)

In [10]:
def build_dataset():
    options = vision.HolisticLandmarkerOptions(
        base_options=python.BaseOptions(
            model_asset_path='../models/holistic_landmarker.task'
        ),
        running_mode=vision.RunningMode.VIDEO,  # VIDEO mode for pre-recorded files
        min_face_detection_confidence=0.5,
        min_pose_detection_confidence=0.5,
        min_hand_landmarks_confidence=0.5,
    )

    skipped   = 0
    processed = 0

    with vision.HolisticLandmarker.create_from_options(options) as landmarker:
        for video_id, val in tqdm(NSLT.items(), desc='Extracting keypoints'):
            class_idx = val['action'][0]
            gloss     = CLASS_TO_GLOSS[class_idx]
            subset    = val['subset']             # 'train' / 'val' / 'test'

            # Save under processed/<subset>/<gloss>/<video_id>.npy
            out_dir  = os.path.join(OUTPUT_DIR, subset, gloss)
            out_path = os.path.join(out_dir, f'{video_id}.npy')
            os.makedirs(out_dir, exist_ok=True)

            if os.path.exists(out_path):
                processed += 1
                continue                          # safe to re-run

            seq = process_video(video_id)
            if seq is not None:
                np.save(out_path, seq)
                processed += 1
            else:
                skipped += 1

    print(f"Done. Processed: {processed} | Skipped: {skipped}")

In [11]:
build_dataset()

Extracting keypoints: 100%|██████████| 2038/2038 [00:00<00:00, 9732.36it/s] 

Done. Processed: 2038 | Skipped: 0
